# Fusión, transiciones y revisión de épocas en Sleep Cassette

¿Se puede mejorar la mezcla de CNN–TCN y LightGBM, y qué épocas conviene revisar primero? Se reutilizan las predicciones fuera de muestra de la [campaña anterior](08_tcn_waveform.ipynb), con las mismas 152.052 épocas, 62 sujetos, 122 noches y cinco folds por sujeto. Los datos corresponden al protocolo retrospectivo con recorte de vigilia asistido por anotaciones. Los 16 sujetos reservados para test quedan fuera del análisis.

- **Rama de descriptores.** Fijar la CNN–TCN y el promedio aritmético 50/50. Comparar el LightGBM de dos EEG y EOG con el LightGBM histórico que también usa la envolvente EMG. El histórico tiene su propio ajuste de hiperparámetros, de modo que la comparación mide el cambio de rama completo.
- **Secuencia.** Aplicar a ambas mezclas el mismo decodificador de Viterbi, con fuerza de transición fijada en 0,25. Contrastar con la decisión independiente por época. La matriz de transición se calcula con los sujetos de entrenamiento de cada fold.
- **Revisión.** Usar la mezcla de referencia de tres canales y un presupuesto fijo del 10 % de cada noche. Comparar entropía, desacuerdo entre modelos y selección aleatoria dentro de la noche.

La métrica primaria de estadificación es F1 macro medio de los cinco folds. Para la revisión se usa la fracción de errores de la mezcla de referencia que cae dentro del presupuesto. Los parámetros de estas comparaciones se fijaron antes de calcular las tablas.

In [1]:
from itertools import product
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score

from sleep_staging.config import CLASSES
from sleep_staging.provenance import hash_file
from sleep_staging.research import paired_subject_bootstrap

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/splits/sc_subjects_v1.csv").is_file())
RUN = ROOT / "var/research/sc-tcn-v1/runs/6e398e69da0dee3125c3"
SPLIT = ROOT / "configs/splits/sc_subjects_v1.csv"
STAGES = np.asarray(CLASSES["task_5"])
COLUMNS = [f"prob_{name}" for name in STAGES]
KEYS = ["epoch_id", "subject_id", "recording_id", "onset_s", "fold", "y_true"]
LAMBDA = 0.25
BUDGET = 0.10

protocol = json.loads((RUN / "protocol.json").read_text())
assert protocol["smoke"] is False and protocol["test_evaluated"] is False
assert hash_file(SPLIT) == protocol["split"]
frozen = pd.read_csv(SPLIT).query("partition == 'development'").set_index("subject_id", verify_integrity=True).fold
inputs = {
    "wave_cnn_tcn_15": "4115c7f5ad2f1def78e48bf96f4efbf6b530242552792b95b50ce9ae276489e4",
    "matched_lightgbm": "d6544c02597431b1e745f4b3af8f41d44197b45d76aa429c855d4ab69f72b2fe",
    "historical_lightgbm_four_channels": "a03fa1b2a5a4243eedcd2e4fb11b097d1a8a237d6974fceb375adf8b760f2bc8",
    "wave_tcn_lightgbm_50_50": "7483c63f7fedeab7e83b769c11c31ab4c0e3c2e071bf9765269b6b5c004570b8",
}
prob = {}
meta = None
for model, sha in inputs.items():
    path = RUN / model / "predictions.parquet"
    assert hash_file(path) == sha, f"Archivo distinto del evaluado en este análisis: {model}"
    df = pd.read_parquet(path).sort_values(["recording_id", "onset_s"]).reset_index(drop=True)
    if meta is None:
        meta = df[KEYS].copy()
    pd.testing.assert_frame_equal(meta, df[KEYS], check_dtype=False)
    p = df[COLUMNS].to_numpy(dtype=np.float64)
    assert np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
    np.testing.assert_allclose(p.sum(axis=1), 1, atol=1e-5)
    np.testing.assert_array_equal(STAGES[p.argmax(axis=1)], df.y_pred)
    prob[model] = p

assert len(meta) == 152_052 and meta.epoch_id.is_unique
assert not meta.duplicated(["recording_id", "onset_s"]).any()
assert set(meta.subject_id) == set(frozen.index) and meta.fold.eq(meta.subject_id.map(frozen)).all()
assert meta.groupby("recording_id").subject_id.nunique().eq(1).all()
assert set(meta.fold) == set(range(5)) and set(meta.y_true) == set(STAGES)

cnn = prob["wave_cnn_tcn_15"]
tree = prob["matched_lightgbm"]
base = prob["wave_tcn_lightgbm_50_50"]
historical_mix = (cnn + prob["historical_lightgbm_four_channels"]) / 2
np.testing.assert_allclose(base, (cnn + tree) / 2, rtol=0, atol=1e-12)
y = pd.Categorical(meta.y_true, categories=STAGES).codes.astype(np.int64)
fold = meta.fold.to_numpy()
K = len(STAGES)
n1 = STAGES.tolist().index("N1")
print(f"{len(meta):,} épocas, {meta.subject_id.nunique()} sujetos y {meta.recording_id.nunique()} noches de desarrollo.")

.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


152,052 épocas, 62 sujetos y 122 noches de desarrollo.


## Decodificación por noche

Para dos estadios consecutivos $i,j$ se añade al logaritmo de la probabilidad por época el término $0{,}25\log(P(j\mid i)/P(j))$. La división por la frecuencia de destino evita volver a sumar una preferencia marginal pura por estadios frecuentes. Se suma un pseudoconteo por celda a la tabla de transiciones de entrenamiento. Ambos términos se estiman sobre los mismos pares consecutivos. La primera época de cada bloque usa solo las probabilidades del clasificador.

Viterbi busca la secuencia completa de mayor puntuación dentro de cada bloque. El algoritmo empieza de nuevo al cambiar de noche o al faltar una época de 30 s. La elección de una época para puntuar sigue la elegibilidad del protocolo anterior. La literatura de estadificación ha explorado el uso de matrices de transición para decodificar hipnogramas. [Estudio sobre decodificación de estadios con Viterbi](https://pubmed.ncbi.nlm.nih.gov/22908930/)

In [2]:
has_prev = (meta.recording_id.eq(meta.recording_id.shift()) & meta.onset_s.diff().eq(30)).to_numpy()
has_next = np.r_[has_prev[1:], False]
change_from_prev = has_prev & (y != np.roll(y, 1))
next_change = np.r_[change_from_prev[1:], False]
boundary = has_prev & has_next & (change_from_prev | next_change)
stable = has_prev & has_next & ~boundary
np.testing.assert_equal((int(boundary.sum()), int(stable.sum())), (26_212, 125_343))
starts = np.flatnonzero(~has_prev)
blocks = list(zip(starts, np.r_[starts[1:], len(y)], strict=True))
night_code, nights = pd.factorize(meta.recording_id, sort=False)

def transition_bonus(labels, contiguous, training):
    # Both members of each labeled pair must belong to training.
    dest = np.flatnonzero(contiguous & training & np.r_[False, training[:-1]])
    pair = K * labels[dest - 1] + labels[dest]
    counts = np.bincount(pair, minlength=K * K).reshape(K, K) + 1.0
    conditional = counts / counts.sum(axis=1, keepdims=True)
    marginal = counts.sum(axis=0) / counts.sum()
    return np.log(conditional / marginal)

def viterbi(unary, pair_scores):
    length, n_classes = unary.shape
    best = unary[0].copy()
    previous = np.zeros((length, n_classes), dtype=int)
    classes = np.arange(n_classes)
    for t in range(1, length):
        choices = best[:, None] + pair_scores
        previous[t] = choices.argmax(axis=0)
        best = unary[t] + choices[previous[t], classes]
    states = np.empty(length, dtype=int)
    states[-1] = best.argmax()
    for t in range(length - 1, 0, -1):
        states[t - 1] = previous[t, states[t]]
    return states

bonus = {f: LAMBDA * transition_bonus(y, has_prev, fold != f) for f in range(5)}
def decode(probabilities):
    unary = np.log(np.maximum(probabilities, 1e-12))
    labels = np.full(len(unary), -1, dtype=int)
    for start, stop in blocks:
        outer = fold[start]
        assert (fold[start:stop] == outer).all()
        labels[start:stop] = viterbi(unary[start:stop], bonus[outer])
    assert (labels >= 0).all()
    return labels

# Verify orientation and backtracking against exhaustive enumeration on a short real sequence.
start = next(a for a, b in blocks if b - a >= 3)
short = np.log(np.maximum(base[start:start + 3], 1e-12))
pair = bonus[fold[start]]
def path_score(states):
    return sum(short[t, s] for t, s in enumerate(states)) + sum(pair[a, b] for a, b in zip(states[:-1], states[1:]))
np.testing.assert_allclose(path_score(viterbi(short, pair)), max(map(path_score, product(range(K), repeat=3))))
np.testing.assert_array_equal(viterbi(short, np.zeros_like(pair)), short.argmax(axis=1))
for f in range(5):
    # Altering evaluation labels cannot affect the estimator used for this fold.
    withheld = np.where(fold == f, (y + 1) % K, y)
    np.testing.assert_array_equal(bonus[f], LAMBDA * transition_bonus(withheld, has_prev, fold != f))

reference = "Tres canales"
replacement = "Árbol histórico con EMG"
decoded_ref = "Tres canales + Viterbi"
decoded_replacement = "Árbol histórico con EMG + Viterbi"
predicted = {reference: base.argmax(axis=1), replacement: historical_mix.argmax(axis=1),
             decoded_ref: decode(base), decoded_replacement: decode(historical_mix)}

## Rendimiento y fragmentación del hipnograma

Cada fold tiene el mismo peso en el F1 macro medio. El F1 de N1 y las exactitudes por proximidad a una transición se calculan sobre todas las épocas externas juntas. Un cambio cuenta cuando dos épocas puntuadas y contiguas tienen estadios distintos. El error absoluto en el número de cambios se promedia sobre las 122 noches.

In [3]:
def changes_per_night(stages):
    change = has_prev & (stages != np.roll(stages, 1))
    return np.bincount(night_code[change], minlength=len(nights))

def macro_f1(truth, decision):
    return f1_score(truth, decision, labels=np.arange(K), average="macro", zero_division=0)

gold_changes = changes_per_night(y)
rows, scores = [], {}
for name, decision in predicted.items():
    scores[name] = np.array([macro_f1(y[fold == f], decision[fold == f]) for f in range(5)])
    counts = changes_per_night(decision)
    rows.append({"Variante": name, "F1 macro CV": scores[name].mean(), "DE folds": scores[name].std(),
                 "F1 N1": f1_score(y == n1, decision == n1),
                 "Acc. transición": (decision[boundary] == y[boundary]).mean(),
                 "Acc. estable": (decision[stable] == y[stable]).mean(),
                 "MAE cambios/noche": np.abs(counts - gold_changes).mean(),
                 "Sesgo cambios/noche": (counts - gold_changes).mean()})
board = pd.DataFrame(rows).set_index("Variante").sort_values("F1 macro CV", ascending=False)
np.testing.assert_allclose(board.loc[reference, "F1 macro CV"], 0.8009455475158577, atol=1e-10)
display(board.round(4))
display(pd.DataFrame(scores).rename_axis("Fold").round(4))

,F1 macro CV,DE folds,F1 N1,Acc. transición,Acc. estable,MAE cambios/noche,Sesgo cambios/noche
Variante,,,,,,,
Árbol histórico con EMG,0.8018,0.0196,0.5437,0.6197,0.8913,26.4754,-0.8197
Tres canales,0.8009,0.0183,0.5401,0.6170,0.8914,27.7541,0.4590
Tres canales + Viterbi,0.7964,0.0204,0.5407,0.5715,0.8921,64.3934,-64.1311
Árbol histórico con EMG + Viterbi,0.7959,0.0212,0.5412,0.5716,0.8913,65.4098,-65.0656


,Tres canales,Árbol histórico con EMG,Tres canales + Viterbi,Árbol histórico con EMG + Viterbi
Fold,,,,
0,0.7654,0.7637,0.7560,0.7539
1,0.8079,0.8093,0.8042,0.8039
2,0.8148,0.8177,0.8059,0.8099
3,0.8133,0.8137,0.8119,0.8087
4,0.8033,0.8047,0.8042,0.8031


## Diferencias emparejadas por sujeto

Se remuestrean 2.000 veces sujetos completos con todas sus noches, manteniendo juntas las predicciones de los dos métodos. El intervalo se refiere a la diferencia de F1 macro agrupado, condicionado a los modelos ajustados. Los contrastes son exploratorios y comparten la misma cohorte de desarrollo.

In [4]:
paired_rows = []
comparisons = [(reference, replacement), (reference, decoded_ref), (replacement, decoded_replacement),
               (reference, decoded_replacement)]
def prediction_frame(decisions):
    return meta[["epoch_id", "subject_id", "y_true"]].assign(y_pred=STAGES[decisions])
for a, b in comparisons:
    boot = paired_subject_bootstrap(prediction_frame(predicted[a]), prediction_frame(predicted[b]))
    paired_rows.append({"Referencia": a, "Alternativa": b, "Δ F1 medio CV": (scores[b] - scores[a]).mean(),
                        "Δ F1 agrupado": boot["delta"], "IC95 inferior": boot["ci95"][0],
                        "IC95 superior": boot["ci95"][1], "Folds con mejora": int((scores[b] > scores[a]).sum())})
paired = pd.DataFrame(paired_rows)
display(paired.round(4))

,Referencia,Alternativa,Δ F1 medio CV,Δ F1 agrupado,IC95 inferior,IC95 superior,Folds con mejora
0,Tres canales,Árbol histórico con EMG,0.0009,0.0008,-0.0011,0.0027,4
1,Tres canales,Tres canales + Viterbi,-0.0045,-0.0047,-0.0069,-0.0027,1
2,Árbol histórico con EMG,Árbol histórico con EMG + Viterbi,-0.0059,-0.0060,-0.0079,-0.0042,0
3,Tres canales,Árbol histórico con EMG + Viterbi,-0.0050,-0.0052,-0.0078,-0.0026,0


## Presupuesto de revisión por noche

Se marca el 10 % de las épocas de cada noche, redondeando hacia abajo. Las prioridades usan solo probabilidades. La primera es la entropía de la mezcla de referencia. La segunda es la divergencia de Jensen–Shannon entre las ramas CNN–TCN y LightGBM. Los empates se ordenan por tiempo. Como control se calcula la esperanza exacta de una selección aleatoria del mismo tamaño dentro de cada noche.

Un error es una discrepancia con el hipnograma de referencia. La captura de errores de N1 usa como denominador las épocas de N1 verdadero clasificadas en otro estadio. La simulación mide localización de discrepancias. El efecto de una segunda lectura humana requeriría observaciones de revisores. [Estudio de incertidumbre y revisión dirigida en estadificación](https://pubmed.ncbi.nlm.nih.gov/34593972/)

In [5]:
def entropy(p):
    p = p / p.sum(axis=1, keepdims=True)
    return -(p * np.log(np.maximum(p, 1e-12))).sum(axis=1)

ent = entropy(base)
jsd = entropy((cnn + tree) / 2) - (entropy(cnn) + entropy(tree)) / 2
groups = list(meta.groupby("recording_id", sort=False).indices.values())
wrong = predicted[reference] != y
wrong_n1 = wrong & (y == n1)
review_rows = []
review_masks = {}
def review_row(name, selected, hits, hits_n1):
    return {"Prioridad": name, "Épocas marcadas": selected, "Fracción revisada": selected / len(y),
            "Errores localizados": hits, "Captura de errores": hits / wrong.sum(),
            "Captura errores en N1 real": hits_n1 / wrong_n1.sum(),
            "Fracción errónea de las marcadas": hits / selected,
            "Exactitud de las restantes": 1 - (wrong.sum() - hits) / (len(y) - selected)}

for name, priority in [("Entropía de la mezcla", ent), ("Desacuerdo entre ramas", jsd)]:
    marked = np.zeros(len(y), dtype=bool)
    for ix in groups:
        n = int(np.floor(BUDGET * len(ix)))
        marked[ix[np.argsort(-priority[ix], kind="stable")[:n]]] = True
    review_masks[name] = marked
    review_rows.append(review_row(name, int(marked.sum()), int(wrong[marked].sum()), int(wrong_n1[marked].sum())))

selected = sum(int(np.floor(BUDGET * len(g))) for g in groups)
expected_hits = sum(int(np.floor(BUDGET * len(g))) * wrong[g].mean() for g in groups)
expected_n1 = sum(int(np.floor(BUDGET * len(g))) * wrong_n1[g].mean() for g in groups)
review_rows.append(review_row("Aleatoria por noche (esperanza)", selected, expected_hits, expected_n1))
review = pd.DataFrame(review_rows).set_index("Prioridad")
print(f"{wrong.sum():,} discrepancias totales de la referencia entre {len(y):,} épocas.")
display(review.round(4))

23,750 discrepancias totales de la referencia entre 152,052 épocas.


,Épocas marcadas,Fracción revisada,Errores localizados,Captura de errores,Captura errores en N1 real,Fracción errónea de las marcadas,Exactitud de las restantes
Prioridad,,,,,,,
Entropía de la mezcla,15154,0.0997,7190.0000,0.3027,0.2344,0.4745,0.8790
Desacuerdo entre ramas,15154,0.0997,5359.0000,0.2256,0.2112,0.3536,0.8657
Aleatoria por noche (esperanza),15154,0.0997,2366.5161,0.0996,0.0996,0.1562,0.8438


## Variación entre sujetos y errores de N1

El intervalo de captura remuestrea los 62 sujetos completos 2.000 veces, conservando el presupuesto de cada noche. La mediana y los cuartiles por sujeto describen la variabilidad individual. La tabla de N1 muestra a qué estadio se asignaron las épocas verdaderas de N1 omitidas por la mezcla de tres canales.

In [6]:
ids, subjects = pd.factorize(meta.subject_id, sort=True)
missed = np.bincount(ids[wrong], minlength=len(subjects))
assert (missed > 0).all()
rng = np.random.default_rng(42)
sampled = rng.integers(0, len(subjects), size=(2000, len(subjects)))
capture_rows = []
for name, marked in review_masks.items():
    caught = np.bincount(ids[wrong & marked], minlength=len(subjects))
    values = caught[sampled].sum(axis=1) / missed[sampled].sum(axis=1)
    low, high = np.quantile(values, [0.025, 0.975])
    q1, median, q3 = np.quantile(caught / missed, [0.25, 0.50, 0.75])
    capture_rows.append({"Método": name, "Captura agrupada": caught.sum() / missed.sum(),
                         "IC95 inferior": low, "IC95 superior": high,
                         "Q1 por sujeto": q1, "Mediana por sujeto": median, "Q3 por sujeto": q3})
display(pd.DataFrame(capture_rows).set_index("Método").round(4))

missed_as = pd.Series(STAGES[predicted[reference][wrong_n1]]).value_counts()
display(missed_as.to_frame("Épocas N1 omitidas").assign(Fracción=missed_as / missed_as.sum()).round(4))
print(f"N1 omitido clasificado como N2 o vigilia: {(missed_as['N2'] + missed_as['W']) / missed_as.sum():.1%}.")

,Captura agrupada,IC95 inferior,IC95 superior,Q1 por sujeto,Mediana por sujeto,Q3 por sujeto
Método,,,,,,
Entropía de la mezcla,0.3027,0.2824,0.3237,0.2648,0.3069,0.3751
Desacuerdo entre ramas,0.2256,0.2120,0.2409,0.1832,0.2346,0.2723


,Épocas N1 omitidas,Fracción
N2,3838,0.5203
W,2165,0.2935
REM,1262,0.1711
N3,112,0.0152


N1 omitido clasificado como N2 o vigilia: 81.4%.


## Interpretación y siguientes experimentos

La rama histórica con EMG obtuvo el mayor F1 macro de desarrollo (0,8018). Su ventaja de F1 agrupado sobre el sistema de tres canales fue de 0,0008, con IC del 95 % de −0,0011 a +0,0027. Conservaría el sistema de tres canales como referencia. Para atribuir una diferencia a EMG habría que repetir los dos entrenamientos bajo las mismas condiciones. En la mezcla de referencia, Viterbi subestimó los cambios entre estadios en 64 por noche en promedio y redujo la exactitud cerca de esos cambios de 61,70 % a 57,15 %.

1. **Representación tiempo–frecuencia.** Entrenar una CNN 2D de log-espectrogramas con los mismos dos EEG, EOG y TCN de 15 épocas. Comparar ondas, espectrogramas y una mezcla fija de 50 % LightGBM, 25 % CNN de ondas y 25 % CNN espectral. Evaluar F1 macro, N1 y las confusiones N1–N2 y N1–vigilia con varias semillas y los mismos folds. [XSleepNet](https://arxiv.org/abs/2007.05492)
2. **Revisión por incertidumbre.** Medir el acuerdo de lectores independientes en muestras de entropía alta y en una muestra aleatoria de control, con los lectores cegados al criterio de selección. Registrar tiempo por noche y cambios en ambas direcciones, incluidos los aciertos del modelo que modifica el revisor. [Revisión dirigida por incertidumbre](https://pubmed.ncbi.nlm.nih.gov/34593972/)
3. **Generalización.** Definir la ventana de entrada sin consultar el hipnograma, congelar el protocolo y medir la transferencia de los tres canales de Cassette a las 44 noches de 22 sujetos de Sleep Telemetry. Informar placebo y temazepam y mantener juntas las dos noches al remuestrear sujetos. [Sleep-EDF Expanded](https://physionet.org/content/sleep-edfx/1.0.0/)

In [7]:
winner = board.index[0]
print(f"Mayor F1 macro de desarrollo: {winner}, {board.loc[winner, 'F1 macro CV']:.4f} ± {board.loc[winner, 'DE folds']:.4f}.")
print(f"Con entropía se localiza el {review.loc['Entropía de la mezcla', 'Captura de errores']:.1%} de las discrepancias.")

Mayor F1 macro de desarrollo: Árbol histórico con EMG, 0.8018 ± 0.0196.
Con entropía se localiza el 30.3% de las discrepancias.
